# Protocol 2 — Active Inference Agent Simulations

Tests predictions Pred 2.A–Pred 2.D from :

* **Pred 2.A** — APGI agent with somatic markers outperforms β-lesion agent on cumulative reward
* **Pred 2.B** — Post-ignition action entropy increases relative to pre-ignition baseline
* **Pred 2.C** — Somatic marker retrieval M̂ temporally precedes threshold crossing
* **Pred 2.D** — β produces specific deficit not seen in other single-parameter lesions

> Computational protocol only — no human participants.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from apgi.core import compute_pi_i_eff, compute_S_t, compute_theta_t, ignition_criterion

## 1 — Somatic marker model and agent parameters

In [ ]:
# Protocol 2 APGI parameters
KAPPA = 100.0
ALPHA = 0.3
BETA = 0.5
GAMMA_V = 0.6
GAMMA_A = 0.3
PI_I_BASE = 1.0
VOLATILITY = [0.1, 0.3, 0.6]  # sigma_env schedule
N_TRIALS = 500
N_AGENTS = 100


def M_hat(V_ca, A_ca):
    """Somatic marker: M_hat(c,a) = gamma_V * V(c,a) + gamma_A * A(c,a)"""
    return GAMMA_V * V_ca + GAMMA_A * A_ca

## 2 — Simulate agent types across volatility levels

In [ ]:
def run_agent(agent_type, sigma_env, rng):
    rewards = []
    theta_t = compute_theta_t(1.0, 0.5, ALPHA, BETA)
    for _ in range(N_TRIALS):
        C = rng.uniform(0.5, 2.0)
        V_info = rng.uniform(0.1, 1.0)
        V_ca = rng.uniform(0.2, 0.8)
        A_ca = rng.uniform(0.1, 0.5)
        if agent_type == "full_apgi":
            pi_i = PI_I_BASE * np.exp(BETA * M_hat(V_ca, A_ca))
        elif agent_type == "beta_lesion":
            pi_i = PI_I_BASE
        elif agent_type == "pi_i_lesion":
            pi_i, C = PI_I_BASE, 0.0
        elif agent_type == "alpha_lesion":
            pi_i, V_info = PI_I_BASE, 0.0
        else:
            rewards.append(rng.uniform(0, 1) * (1 - sigma_env))
            continue
        pi_i_eff = compute_pi_i_eff(pi_i, C, kappa=KAPPA)
        S_t = compute_S_t(
            rng.uniform(0.8, 1.5),
            rng.uniform(0.2, 1.0),
            pi_i_eff,
            rng.uniform(0.1, 0.8),
        )
        ignited = ignition_criterion(S_t, theta_t)
        theta_t = compute_theta_t(C, V_info, ALPHA, BETA)
        rewards.append(
            float(ignited) * rng.binomial(1, max(0.1, 0.7 - sigma_env * 0.5))
        )
    return np.mean(rewards)


rng = np.random.default_rng(42)
agents = ["full_apgi", "beta_lesion", "pi_i_lesion", "alpha_lesion", "random"]
results = {
    a: {s: [run_agent(a, s, rng) for _ in range(N_AGENTS)] for s in VOLATILITY}
    for a in agents
}
print("Mean rewards (sigma=0.3):")
for a in agents:
    print(f"  {a:<18} {np.mean(results[a][0.3]):.3f}")

## 3 — Prediction 2.A: Reward advantage across volatility levels

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
agent_labels = {
    "full_apgi": "Full APGI",
    "beta_lesion": "β-lesion",
    "pi_i_lesion": "Π⁺-lesion",
    "alpha_lesion": "α-lesion",
    "random": "Random",
}
colors = ["#2166ac", "#d6604d", "#9966FF", "#FFCC00", "#AAAAAA"]

for ax, sigma in zip(axes, VOLATILITY):
    means = [np.mean(results[a][sigma]) for a in agents]
    sems = [np.std(results[a][sigma]) / np.sqrt(N_AGENTS) for a in agents]
    ax.bar(
        range(len(agents)),
        means,
        yerr=sems,
        color=colors,
        alpha=0.85,
        edgecolor="white",
        capsize=3,
        width=0.6,
    )
    ax.set_xticks(range(len(agents)))
    ax.set_xticklabels(
        [agent_labels[a] for a in agents], rotation=25, ha="right", fontsize=8
    )
    ax.set_title(f"σ_env = {sigma}", fontsize=11)
    ax.set_ylabel("Mean reward" if sigma == 0.1 else "", fontsize=10)
    ax.set_ylim(0, 0.7)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Pred 2.A — Somatic Marker Agent Reward Advantage", fontsize=12)
plt.tight_layout()
plt.show()

## 4 — Prediction 2.D: β-lesion deficit is specific

The somatic marker pathway (β) confers an advantage that is not replicated by removing Πⁱ or α:

| Agent | Mechanism disabled |
|---|---|
| Full APGI | None — baseline |
| β-lesion | Somatic marker modulation (M̂ → 0) |
| Πⁱ-lesion | Metabolic gating (Πⁱ_eff held constant) |
| α-lesion | Metabolic cost weighting (α = 0) |
| Random | No APGI mechanism |

In [ ]:
sigma_test = 0.3
means_test = {a: np.mean(results[a][sigma_test]) for a in agents}
full = means_test["full_apgi"]
for a in agents[1:]:
    deficit = full - means_test[a]
    print(
        f"  {agent_labels[a]:<18}  reward={means_test[a]:.3f}  deficit vs full APGI={deficit:+.3f}"
    )